In [1]:
!git clone https://github.com/huutrank4ds/ViFlow.git

import sys
sys.path.append("/kaggle/working/ViFlow") 

# Trỏ chính xác vào file requirements.txt bên trong thư mục vừa clone
!pip install -r ViFlow/requirements.txt -q
!pip install pyngrok -q

Cloning into 'ViFlow'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 94 (delta 41), reused 70 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 1.34 MiB | 23.24 MiB/s, done.
Resolving deltas: 100% (41/41), done.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 76.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.7 MB/s eta 0:00:00


In [ ]:
# Import các local modules của hệ thống
from engine import ViFlowEngine
from dataset import ViFlowProcessor
from inference import get_model, get_bigvgan_vocoder
import yaml
import torch
from kaggle_secrets import UserSecretsClient


with open("ViFlow/configs.yaml", "r") as f:
    config = yaml.safe_load(f)

# =================================================================
MODEL_CHECKPOINT = "/kaggle/input/models/huutrank4ds/viflow-full-gear/pytorch/default/10/viflow_epoch_112.pt"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
viflow_processor = ViFlowProcessor(config['mel'], config['model']['vocab_path'], DEVICE)


# =================================================================
viflow_engine = ViFlowEngine()
viflow_model = get_model(MODEL_CHECKPOINT, viflow_processor.phoneme_tokenizer.vocab_size, DEVICE, config)
vocoder = get_bigvgan_vocoder(HF_TOKEN, DEVICE)

🔊 Đang khởi tạo bộ chuyển đổi ngôn ngữ sea_g2p cho Tiếng Việt...
Đang load tập vocab...
Đã load vocab với vocab_size là 71
🛠️ TextEncoder: Sử dụng 4 lớp ConvNeXtV2
Đang đọc trọng số từ file checkpoint /kaggle/input/models/huutrank4ds/viflow-full-gear/pytorch/default/10/viflow_epoch_112.pt!
Nạp trọng số mô hình thành công!


config.json: 0.00B [00:00, ?B/s]

bigvgan_generator.pt:   0%|          | 0.00/450M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...


In [ ]:
import gradio as gr
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import librosa # Cần import thêm librosa
import torchaudio
from dataset import ViFlowProcessor

# ==========================================
# 1. HÀM INFERENCE CỐT LÕI
# ==========================================
def synthesize_voice(audio_ref, text_ref, text_target, solver, cfg_scale, steps, speed):
    if not audio_ref or not text_ref or not text_target:
        return None, "⚠️ Vui lòng cung cấp đủ Audio mồi, Text mồi và Text cần sinh!"
    
    try:
        # Xử lý Audio mồi -> Mel Spectrogram
        wav, sr = torchaudio.load(audio_ref)
        if sr != 24000:
            wav = torchaudio.transforms.Resample(sr, 24000)(wav)
        
        input_processed = viflow_processor.prepare_input(wav, text_ref, text_target, speed)
        x0 = input_processed['x0']
        cond = input_processed['cond']
        text_ids = input_processed['text_ids']
        target_mask = input_processed['target_mask']
        n_prompt = input_processed['n_prompt']
        
        # Giải ODE & Sinh âm thanh
        with torch.no_grad():
            x_gen = viflow_engine.solve_ode(
                model=viflow_model, x0=x0, steps=int(steps), cond=cond, text_ids=text_ids,
                mel_mask=None, target_mask=target_mask, 
                sway_coef=-1.0, cfg_scale=cfg_scale, solver=solver
            )
            
            if x_gen.dim() == 4: x_gen = x_gen.squeeze(-1)
            
            # Cắt bỏ phần mồi, chỉ giữ phần sinh ra
            x_gen_target = x_gen[:, n_prompt:, :]
            
            gen_wav_tensor = vocoder(x_gen_target.transpose(1, 2)).squeeze()
            max_val = gen_wav_tensor.abs().max()
            if max_val > 0.95:
                gen_wav_tensor = (gen_wav_tensor / max_val) * 0.95

        return (24000, gen_wav_tensor.cpu().numpy()), "✅ Tổng hợp thành công!"
        
    except Exception as e:
        import traceback
        traceback.print_exc()
        return None, f"❌ Có lỗi xảy ra: {str(e)}"

# Xây dựng giao diện Gradio
with gr.Blocks(title="ViFlow Voice Cloning", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ ViFlow - Hệ Thống Sao Chép Giọng Nói Tiếng Việt")
    
    with gr.Row():
        with gr.Column():
            audio_in = gr.Audio(type="filepath", label="1. Tải lên Audio tham chiếu (Giọng mồi)")
            text_ref_in = gr.Textbox(lines=2, label="2. Văn bản của đoạn Audio trên")
            text_target_in = gr.Textbox(lines=3, label="3. Văn bản mục tiêu (Câu cần sinh)")
            
            with gr.Accordion("Cài đặt nâng cao (Advanced)", open=False):
                solver_in = gr.Dropdown(choices=["heun", "euler"], value="heun", label="Thuật toán giải ODE")
                cfg_in = gr.Slider(minimum=1.0, maximum=5.0, value=2.0, step=0.1, label="CFG Scale")
                steps_in = gr.Slider(minimum=4, maximum=64, value=16, step=1, label="ODE Steps")
                speed_in = gr.Slider(minimum=0.5, maximum=2.0, value=1.0, step=0.1, label="Tốc độ đọc")
                
            submit_btn = gr.Button("🚀 BẮT ĐẦU TỔNG HỢP", variant="primary")
            
        with gr.Column():
            audio_out = gr.Audio(label="🎧 Kết quả âm thanh")
            status_out = gr.Textbox(label="Trạng thái", interactive=False)

    submit_btn.click(
        fn=synthesize_voice,
        inputs=[audio_in, text_ref_in, text_target_in, solver_in, cfg_in, steps_in, speed_in],
        outputs=[audio_out, status_out]
    )

# Mở cổng Ngrok và Launch
if __name__ == "__main__":
    try:
        NGROK_TOKEN = user_secrets.get_secret("NGROK_AUTHTOKEN")
        ngrok.set_auth_token(NGROK_TOKEN)
        ngrok.kill() 
        public_url = ngrok.connect(7860).public_url
        print(f"🌍 TRUY CẬP GIAO DIỆN WEB TẠI ĐÂY: {public_url}")
    except Exception as e:
        print("⚠️ Chạy cục bộ vì lỗi ngrok.")
    demo.launch(server_name="0.0.0.0", server_port=7860, quiet=False)
    print("Server đang chạy, đừng tắt cell này!")
    import time
    while True:
        time.sleep(1)

/tmp/ipykernel_58/176790333.py:54: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="ViFlow Voice Cloning", theme=gr.themes.Soft()) as demo:


🌍 TRUY CẬP GIAO DIỆN WEB TẠI ĐÂY: https://5140-34-145-28-120.ngrok-free.app
* Running on local URL:  http://0.0.0.0:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://7befe34eafeac18a62.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Server đang chạy, đừng tắt cell này!
